# Pesquisa de Subenquadramento de Engenharia — PNCP

Modelo para identificar contratos rotulados como **"serviços gerais"** que na
verdade são **engenharia/obras** (subenquadramento, Lei 14.133/2021).

## Princípio metodológico (PU Learning)

Os rótulos `engenharia` e `obras` da coleta são **confiáveis** — quando o órgão
marcou assim, é porque é. Só o rótulo `geral` é **ruidoso**: parte é serviço
comum de verdade, parte é engenharia mal classificada. O estudo usa os
confirmados como âncora para encontrar engenharia escondida no `geral`.

## Roteiro

| Etapa | O que faz |
|---|---|
| 0 | Setup + relatório vivo |
| 1 | Carregar base + pré-processamento + EDA mínima |
| 2 | Embeddings SBERT + **filtro PU** (centróide eng+obras) |
| 3 | **Clusterização auto-k** (6–12, escolhe o mais coeso) |
| 4 | **Revisão humana** assistida por LLM (você decide) |
| 5 | LLM descreve perfis eng vs não-eng |
| 6 | Treina classificador (LogReg + Random Forest) |
| 7 | **Pontua TODOS os `geral`** → ranking de suspeitos |
| 8 | Validação manual (~80 contratos) → precisão/recall |
| 9 | Visualização UMAP 2D |
| 10 | Veredito LLM nos suspeitos |
| 11 | **Análise de rito** (TR/Edital da licitação → ART/CREA) |
| 12 | Exportação (modelo + relatório) |

**Autossuficiente:** todo o código está neste notebook. Não precisa clonar nem
instalar pacote nenhum além das libs públicas. Apenas `contratos.parquet` no
Drive.

## 0. Setup

Bibliotecas, Drive, LLM e o **relatório vivo** (cada etapa atualiza um `.md` no
Drive automaticamente — você sempre tem o estado atual).

In [ ]:
!pip install -q sentence-transformers scikit-learn umap-learn nltk networkx
!pip install -q google-generativeai pymupdf pyarrow pandas matplotlib seaborn

In [ ]:
import os, re, time, json, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import datetime as _dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for _p in ('punkt', 'punkt_tab', 'stopwords', 'rslp'):
    nltk.download(_p, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score,
                              silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 160)
print('imports OK')

In [ ]:
# Drive + paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

CAMINHO_PARQUET  = '/content/drive/MyDrive/PNCP_TCC/dados/coleta/contratos.parquet'
PASTA_RESULTADOS = '/content/drive/MyDrive/PNCP_TCC/resultados_pesquisa'
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
print('Resultados em:', PASTA_RESULTADOS)

In [ ]:
# ── LLM local: Ollama + Llama 3.1 na GPU (sem rate limit, sem deprecação) ──
# Você disse que vai usar GPU do Colab — Ollama local é o ideal: não tem 429
# nem o aviso de descontinuação do google.generativeai.
import subprocess, requests

def _ollama_no_ar(porta=11434):
    try:
        return requests.get(f'http://127.0.0.1:{porta}/api/tags', timeout=2).ok
    except Exception:
        return False

if not _ollama_no_ar():
    print('Instalando Ollama (1x por sessão)...')
    subprocess.run('apt-get -qq install -y lshw pciutils zstd >/dev/null 2>&1',
                   shell=True)
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True)
    subprocess.Popen(['ollama', 'serve'],
                     stdout=open('/tmp/ollama.log', 'w'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if _ollama_no_ar():
            break
        time.sleep(1)

OLLAMA_MODELO = 'llama3.1'        # 8B; cabe na T4/L4. Use 'llama3.1:8b-instruct-q4_K_M' p/ mais leve
if _ollama_no_ar():
    subprocess.run(['ollama', 'pull', OLLAMA_MODELO], check=False)
    print('Ollama OK:', _ollama_no_ar(), '| modelo:', OLLAMA_MODELO)
else:
    print('⚠ Ollama não subiu — veja /tmp/ollama.log')

INTERVALO_LLM_SEG = 0             # local: sem espera entre chamadas

def llm_task(system, prompt, max_tokens=800, temperatura=0.2, **kw):
    try:
        r = requests.post('http://127.0.0.1:11434/api/chat', timeout=180,
            json={'model': OLLAMA_MODELO, 'stream': False,
                  'messages': [{'role': 'system', 'content': system},
                               {'role': 'user', 'content': prompt}],
                  'options': {'temperature': temperatura,
                              'num_predict': max_tokens}})
        return r.json().get('message', {}).get('content', '').strip() if r.ok else None
    except Exception as e:
        print('[llm]', str(e)[:120]); return None

def extrair_json(txt):
    if not txt:
        return None
    t = txt.replace('```json', '').replace('```', '')
    m = re.search(r'\{.*\}', t, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

# Teste rápido
_t = llm_task('Responda em uma palavra.', 'Diga OK.')
print('Teste LLM →', _t or '(sem resposta)')

# ── Alternativa: Google Gemini (pacote NOVO google.genai, sem deprecação) ──
# Use se preferir nuvem. Tem rate limit no free; ative billing no AI Studio.
# !pip install -q google-genai
# from google import genai as ggenai
# from google.colab import userdata
# _client = ggenai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
# def llm_task(system, prompt, max_tokens=800, temperatura=0.2, **kw):
#     try:
#         r = _client.models.generate_content(
#             model='gemini-2.5-flash',
#             contents=f'{system}\n\n{prompt}',
#             config={'temperature': temperatura, 'max_output_tokens': max_tokens})
#         return (r.text or '').strip()
#     except Exception as e:
#         print('[llm]', str(e)[:120]); return None
# INTERVALO_LLM_SEG = 4

In [ ]:
# ── Relatório vivo ──────────────────────────────────────────────────────
# Cada etapa chama add_md(secao, conteudo) e o .md é regravado no Drive.
RELATORIO_SECOES = {}
RELATORIO_ORDEM = []
RELATORIO_PATH = f'{PASTA_RESULTADOS}/relatorio_vivo.md'

def add_md(secao, conteudo):
    if secao not in RELATORIO_SECOES:
        RELATORIO_ORDEM.append(secao)
    RELATORIO_SECOES[secao] = conteudo
    linhas = ['# Pesquisa de Subenquadramento — Relatório',
              f'_Atualizado em {_dt.datetime.now():%Y-%m-%d %H:%M}_', '']
    for s in RELATORIO_ORDEM:
        linhas += [f'## {s}', RELATORIO_SECOES[s], '']
    Path(RELATORIO_PATH).write_text('\n'.join(linhas), encoding='utf-8')
    print('  → relatório vivo atualizado')

print('Relatório vivo:', RELATORIO_PATH)

## 1. Base + pré-processamento + EDA mínima

Carrega `contratos.parquet` e define as funções de texto (tudo inline). EDA só
o essencial: distribuição dos rótulos e top termos (para entender a base).

In [ ]:
# ── Funções de pré-processamento (inline) ───────────────────────────────
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))
# Termos burocráticos que aparecem em quase todo contrato e não discriminam.
STOP_DOMINIO = {
    'contratacao','contratacoes','contratar','contrato','contratos','servico',
    'servicos','prestacao','prestar','prestados','prestada','empresa','empresas',
    'especializada','especializado','fornecimento','fornecer','aquisicao',
    'aquisicoes','fornecedor','registro','preco','precos','atender','atendimento',
    'executar','execucao','realizacao','realizar','objeto','objetos','conforme',
    'demanda','demandas','referente','pregao','licitacao','edital','editais',
    'ata','atas','municipio','municipal','estadual','federal','lote','lotes',
    'item','itens',
}
STOP_TUDO = STOP_PT | STOP_DOMINIO
STEMMER = RSLPStemmer()

def _sem_acento(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s).lower())
                    if not unicodedata.combining(c))

def meu_tokenizador(doc):
    toks = word_tokenize(_sem_acento(doc), language='portuguese')
    return [STEMMER.stem(w) for w in toks
            if w not in STOP_TUDO and w.isalnum() and not w.isdigit() and len(w) > 2]

# Remove prefixos burocráticos antes do SBERT — sem isso, "CONTRATAÇÃO DE
# EMPRESA ESPECIALIZADA PARA PRESTAÇÃO DE SERVIÇOS DE..." idêntico em quase
# todo contrato arrasta a similaridade para cima e arruína a discriminação.
_RX_BP = [re.compile(p, re.IGNORECASE) for p in [
    r'^\s*registro\s+de\s+pre[çc]os?\s+(para\s+a?\s*)?',
    r'^\s*contrata[çc][ãa]o\s+(de\s+)?(empresa\s+)?(especializada\s+)?'
    r'(em\s+|para\s+(o?\s+|a?\s+)?(presta[çc][ãa]o\s+(de\s+)?)?(servi[çc]os?\s+(de\s+)?)?)?',
    r'^\s*contrata[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*presta[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*aquisi[çc][ãa]o\s+de\s+', r'^\s*fornecimento\s+de\s+',
    r'^\s*execu[çc][ãa]o\s+de\s+(servi[çc]os?\s+(de\s+)?)?',
    r'^\s*loca[çc][ãa]o\s+de\s+',
]]
def limpar_boilerplate(t):
    if not isinstance(t, str):
        return ''
    s = t.strip()
    for _ in range(3):
        for rx in _RX_BP:
            s = rx.sub('', s, count=1)
    return re.sub(r'\s+', ' ', s).strip()

print('funções de texto definidas')

In [ ]:
# ── Carrega base ────────────────────────────────────────────────────────
df_full = pd.read_parquet(CAMINHO_PARQUET)
COL_UF = next((c for c in df_full.columns
               if c.lower() in ('uf','ufsigla','siglauf','unidadefederativa')), None)
if COL_UF and COL_UF != 'uf':
    df_full = df_full.rename(columns={COL_UF: 'uf'})

cols = [c for c in ['numeroControlePNCP','objeto','rotulo','razaoSocialOrgao',
                     'municipioNome','valor','uf','numeroControlePNCPCompra',
                     'sequencialCompra','anoCompra'] if c in df_full.columns]
df = df_full[cols].copy().rename(columns={'objeto': 'text'})
del df_full
df = df.dropna(subset=['text'])
df = df[df['text'].str.len() > 20].reset_index(drop=True)

# Amostragem opcional (em GPU paga, suba ou use tudo)
N_AMOSTRA = 30_000
if len(df) > N_AMOSTRA:
    df = df.sample(n=N_AMOSTRA, random_state=42).reset_index(drop=True)

print(f'Base de trabalho: {len(df):,} contratos')
print(df['rotulo'].value_counts().to_string())

In [ ]:
# ── EDA mínima: distribuição + top termos ───────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.2))
cont = df['rotulo'].value_counts()
cores = {'engenharia':'#d62728','obras':'#ff7f0e','geral':'#9aa0a6'}
ax.bar(cont.index, cont.values, color=[cores.get(c,'#1f77b4') for c in cont.index])
for c, v in zip(cont.index, cont.values):
    ax.text(c, v, f'{v:,}', ha='center', va='bottom')
ax.set_title(f'Distribuição dos rótulos (n={len(df):,})'); ax.set_ylabel('nº')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/01_distribuicao.png', dpi=120, bbox_inches='tight')
plt.show()

vsm = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                      min_df=5, ngram_range=(1,2), max_features=20000)
Xtf = vsm.fit_transform(df['text'])
top = (pd.DataFrame({'termo': vsm.get_feature_names_out(),
                     'peso': Xtf.toarray().sum(axis=0)})
       .sort_values('peso', ascending=False).head(25))
print('\nTop 25 termos/bigramas:')
print(top.to_string(index=False))

add_md('1. Base',
f'''- Contratos analisados: **{len(df):,}**
- Distribuição: {", ".join(f"{k}={v:,}" for k,v in cont.items())}
- Gráfico: `01_distribuicao.png`''')

## 2. Embeddings SBERT + filtro PU

**SBERT** transforma cada objeto em vetor semântico. Depois, usamos os
**confirmados (eng+obras)** como âncora: calculamos o centróide deles e medimos
a proximidade de cada `geral`. Só os `geral` próximos entram na análise — o
resto é claramente "não-engenharia".

- `eng+obras` → `rotulo_auto = 'eng_obra'` (positivos certeiros)
- `geral` próximo do centróide → **candidato** (vai para clusterização)
- `geral` distante → `rotulo_auto = 'nao'` (negativo confirmado automático)

In [ ]:
# 2.1 Embeddings
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')

print('Gerando embeddings...')
textos = df['text'].map(limpar_boilerplate).tolist()
X_emb = sbert.encode(textos, show_progress_bar=True, batch_size=64,
                     convert_to_numpy=True, normalize_embeddings=True)
print('Embeddings:', X_emb.shape)

In [ ]:
# 2.2 Filtro PU
mask_pos   = df['rotulo'].isin(['engenharia', 'obras']).values
mask_geral = (df['rotulo'] == 'geral').values

centroide = X_emb[mask_pos].mean(axis=0)
centroide = centroide / (np.linalg.norm(centroide) + 1e-12)
sim = X_emb @ centroide          # X_emb já normalizado → produto = cosseno
df['sim_centroide'] = sim

PCT_GERAL_CANDIDATO = 0.30       # top 30% dos geral mais próximos = candidatos
limiar = float(np.quantile(sim[mask_geral], 1 - PCT_GERAL_CANDIDATO))

df['eh_candidato'] = mask_pos | (mask_geral & (sim >= limiar))
df['rotulo_auto'] = ''
df.loc[mask_pos, 'rotulo_auto'] = 'eng_obra'
df.loc[mask_geral & (sim < limiar), 'rotulo_auto'] = 'nao'

n_pos = int(mask_pos.sum())
n_cand_g = int((mask_geral & (sim >= limiar)).sum())
n_nao = int((mask_geral & (sim < limiar)).sum())
print(f'Positivos certeiros (eng+obras): {n_pos:,}')
print(f'Candidatos do geral (próximos):  {n_cand_g:,}')
print(f'Geral → "nao" automático:        {n_nao:,}')
print(f'Limiar de similaridade: {limiar:.3f}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(sim[mask_pos], bins=50, alpha=.7, density=True, color='#1b7837',
        label=f'eng+obras (n={n_pos:,})')
ax.hist(sim[mask_geral], bins=50, alpha=.5, density=True, color='#9aa0a6',
        label=f'geral (n={int(mask_geral.sum()):,})')
ax.axvline(limiar, color='red', ls='--', lw=1.5,
           label=f'limiar (top {PCT_GERAL_CANDIDATO:.0%} dos geral)')
ax.set_xlabel('similaridade ao centróide eng+obras'); ax.set_ylabel('densidade')
ax.set_title('Filtro PU: confirmados vs. geral'); ax.legend()
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/02_filtro_pu.png', dpi=120, bbox_inches='tight')
plt.show()

add_md('2. Filtro PU',
f'''- Positivos certeiros (eng+obras): **{n_pos:,}**
- Candidatos do geral (top {PCT_GERAL_CANDIDATO:.0%}, próximos do centróide): **{n_cand_g:,}**
- Geral classificado como "nao" automaticamente: **{n_nao:,}**
- Limiar de similaridade: {limiar:.3f}
- Gráfico: `02_filtro_pu.png`''')

## 3. Clusterização auto-k (poucos e coesos)

KMeans sobre os **candidatos** (eng+obras + geral próximo). Testamos k=6..12 e
escolhemos o de melhor **silhouette** — garante poucos clusters e bem
separados. Cada cluster mostra o **% de positivos certeiros**: quanto maior,
mais o cluster "é" engenharia.

In [ ]:
# 3.1 Escolhe melhor k por silhouette
mask_cand = df['eh_candidato'].values
X_cand = X_emb[mask_cand]
idx_cand = np.where(mask_cand)[0]
print(f'Candidatos a clusterizar: {len(idx_cand):,}')

rng = np.random.RandomState(42)
amostra_sil = rng.choice(len(X_cand), size=min(5000, len(X_cand)), replace=False)

resultados_k = []
for k in range(6, 13):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X_cand)
    s = silhouette_score(X_cand[amostra_sil], lab[amostra_sil], metric='cosine')
    resultados_k.append((k, s, km, lab))
    print(f'  k={k}: silhouette={s:.3f}')

melhor_k, melhor_sil, melhor_km, melhor_lab = max(resultados_k, key=lambda r: r[1])
print(f'\n🏆 Melhor k = {melhor_k} (silhouette={melhor_sil:.3f})')

df['cluster'] = -1
df.loc[idx_cand, 'cluster'] = melhor_lab

In [ ]:
# 3.2 Descritores de cada cluster (TF-IDF) + pureza de positivos
df_cand = df[df['eh_candidato']].copy()
vsm_d = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                        min_df=3, ngram_range=(1,2), max_features=30000)
Xd = vsm_d.fit_transform(df_cand['text'])
nomes = vsm_d.get_feature_names_out()

desc = []
for c in sorted(df_cand['cluster'].unique()):
    m = (df_cand['cluster'] == c).values
    sub = df_cand[m]
    soma = Xd[m].sum(axis=0).A1
    top_termos = ', '.join(nomes[i] for i in soma.argsort()[::-1][:8])
    d = sub['rotulo'].value_counts().to_dict()
    n_cert = d.get('engenharia',0) + d.get('obras',0)
    desc.append({'cluster': c, 'n_docs': len(sub), 'top_termos': top_termos,
                 'n_certeiros': n_cert, 'n_geral_cand': d.get('geral',0),
                 'pct_certeiros': n_cert/max(1,len(sub))})
df_desc = pd.DataFrame(desc).sort_values('pct_certeiros', ascending=False)
print('Clusters (ordenados por % de positivos certeiros):\n')
print(df_desc[['cluster','n_docs','n_certeiros','n_geral_cand',
               'pct_certeiros','top_termos']].to_string(index=False))

add_md('3. Clusterização',
f'''- Melhor k = **{melhor_k}** (silhouette {melhor_sil:.3f})
- Candidatos clusterizados: {len(df_cand):,}

| cluster | n | % certeiros | top termos |
|---|---|---|---|
''' + '\n'.join(
    f"| {r['cluster']} | {r['n_docs']} | {r['pct_certeiros']:.0%} | {r['top_termos'][:55]} |"
    for _, r in df_desc.iterrows()))

In [ ]:
# 3.3 Visualização dos clusters — barras de tamanho e % de certeiros
# (Boa prática de viz: ordenar por valor, rotular direto na barra, cor com
#  significado — verde = mais "engenharia", cinza = menos.)
import matplotlib.cm as _cm

dfp = df_desc.sort_values('pct_certeiros', ascending=True)   # asc p/ barh
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 0.6 * len(dfp) + 1.5))

# A) % de positivos certeiros por cluster (cor proporcional ao %)
cores = _cm.RdYlGn(dfp['pct_certeiros'].clip(0, 1).values)
axA.barh(dfp['cluster'].astype(str), dfp['pct_certeiros'] * 100, color=cores)
for y, (v, n) in enumerate(zip(dfp['pct_certeiros'], dfp['n_docs'])):
    axA.text(v * 100 + 1, y, f'{v:.0%}', va='center', fontsize=9)
axA.set_xlabel('% de eng+obras confirmados no cluster')
axA.set_ylabel('cluster'); axA.set_title('Pureza de cada cluster (candidatos)')
axA.set_xlim(0, 105)

# B) Tamanho dos clusters (nº de contratos)
axB.barh(dfp['cluster'].astype(str), dfp['n_docs'], color='#4c72b0')
for y, n in enumerate(dfp['n_docs']):
    axB.text(n, y, f' {n:,}', va='center', fontsize=9)
axB.set_xlabel('nº de contratos'); axB.set_title('Tamanho de cada cluster')

plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/03_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

# Mostra a tabela de descritores também (head visível)
print('Descritores dos clusters:')
display(df_desc.reset_index(drop=True))

## 4. Rótulos de treino (automáticos — sem rotulação manual)

Você notou: os clusters **não são** 100% engenharia nem 100% não-eng (só o
cluster mais puro chega perto). Então **não rotulamos clusters à mão**. Em vez
disso, treinamos só com os **certeiros**:

- **positivos** = `engenharia` + `obras` (o rótulo do órgão é confiável)
- **negativos** = `geral` **distante** do centróide de engenharia (o filtro PU
  já separou — são quase com certeza não-eng: seguros, eventos, limpeza...)
- os `geral` **candidatos** (próximos do centróide) ficam de fora do treino —
  são exatamente o que o modelo vai **classificar**.

A clusterização da etapa 3 continua, mas só para você **entender** a base. A
LLM (via Ollama) pode descrever cada cluster, mas isso é informativo.

In [ ]:
# 4.1 Rótulos de treino AUTOMÁTICOS (sem rotulação manual de cluster)
# Você observou que os clusters NÃO são 100% eng ou 100% não-eng — então não
# dá para rotulá-los com certeza. A solução: treinar só com os CERTEIROS:
#   • positivos  = engenharia + obras  (rótulo do órgão é confiável)
#   • negativos  = "geral" DISTANTE do centróide eng (filtro PU = quase certo
#                  que não é engenharia: limpeza, eventos, seguros, etc.)
# Os "geral" candidatos (próximos do centróide) NÃO são rotulados — são
# justamente o que o modelo vai CLASSIFICAR depois.
#
# A clusterização (etapa 3) continua, mas só para ENTENDER/visualizar os
# candidatos — não decide rótulo.

df['rotulo_treino'] = np.nan
df.loc[df['rotulo'].isin(['engenharia', 'obras']), 'rotulo_treino'] = 'eng_obra'
df.loc[(df['rotulo'] == 'geral') & (df['rotulo_auto'] == 'nao'),
       'rotulo_treino'] = 'nao'
# (os candidatos geral ficam com rotulo_treino = NaN → não entram no treino)

n_p = int((df['rotulo_treino'] == 'eng_obra').sum())
n_n = int((df['rotulo_treino'] == 'nao').sum())
n_cand = int(df['rotulo_treino'].isna().sum())
print('Conjunto de TREINO (certeiros automáticos):')
print(f'  positivos (eng+obras):        {n_p:,}')
print(f'  negativos (geral distante):   {n_n:,}')
print(f'  candidatos a classificar:     {n_cand:,} (NÃO entram no treino)')
print('\\nResumo dos clusters dos candidatos (só para você entender a base):')
print(df_desc[['cluster','n_docs','n_certeiros','pct_certeiros',
               'top_termos']].to_string(index=False))

In [ ]:
# 4.2 (OPCIONAL) LLM descreve cada cluster — só informativo, não rotula nada.
# Com Ollama local isto funciona sem rate limit. Pule se não quiser.
SYS_CLUSTER = '''Você é engenheiro especialista em contratações públicas
(Lei 14.133/2021). Recebe termos frequentes e exemplos de um grupo de
contratos. Resuma o que o grupo contém e diga se PARECE engenharia/obras
(execução de serviço técnico CREA/ART ou intervenção física) ou não.
Arquitetura (CAU/RRT) não conta como engenharia.
Responda JSON: {"resumo":"1 frase","parece_eng":"sim"|"nao"|"misto"}'''

FAZER_DESC_LLM = True   # mude para False para pular
desc_clusters = {}
if FAZER_DESC_LLM:
    for c in sorted(df_cand['cluster'].unique()):
        sub = df_cand[df_cand['cluster'] == c]
        drow = df_desc[df_desc['cluster'] == c].iloc[0]
        amostras = '\\n'.join(f'- {str(o)[:180]}'
                              for o in sub['text'].sample(min(8, len(sub)), random_state=c))
        p = extrair_json(llm_task(SYS_CLUSTER,
              f"Termos: {drow['top_termos']}\\n\\nExemplos:\\n{amostras}")) or {}
        desc_clusters[c] = p
        print(f"cluster {c} ({drow['pct_certeiros']:.0%} certeiros): "
              f"{p.get('parece_eng','?'):5s} — {p.get('resumo','')[:75]}")
        if INTERVALO_LLM_SEG:
            time.sleep(INTERVALO_LLM_SEG)
else:
    print('Descrição por LLM pulada (FAZER_DESC_LLM=False).')

> ℹ️ Não há rotulação manual de clusters nesta versão — os rótulos de treino
> são automáticos (certeiros). Siga direto para o treino.

In [ ]:
# 4.3 Registra no relatório
add_md('4. Rótulos de treino (automáticos)',
f'''Sem rotulação manual de clusters (eles são mistos). Treino com certeiros:
- positivos (engenharia+obras): **{n_p:,}**
- negativos (geral distante do centróide): **{n_n:,}**
- candidatos a classificar (geral próximo): **{n_cand:,}**

Descrição dos clusters dos candidatos:

''' + df_desc[['cluster','n_docs','pct_certeiros','top_termos']].to_markdown(index=False))
print('registrado no relatório vivo')

## 5. LLM descreve os perfis (documentação)

A partir dos clusters certeiros, a LLM sintetiza o perfil de cada classe —
padrões léxicos e tipos de serviço. Serve para documentar e auditar.

In [ ]:
SYS_PERFIL = '''Você é engenheiro analista de contratações públicas. Recebe
objetos de contratos de um grupo. Descreva o perfil em JSON:
{"resumo": "1-2 frases",
 "padroes_lexicais": ["termo1","termo2","termo3"],
 "tipos_servico": ["tipo1","tipo2"]}'''

def perfil(rot, n=15):
    sub = df[df['rotulo_treino'] == rot]
    if not len(sub):
        return {}
    objs = '\n'.join(f'- {str(o)[:200]}'
                      for o in sub['text'].sample(min(n, len(sub)), random_state=1))
    for _ in range(2):   # uma tentativa extra se a 1a vier vazia
        out = extrair_json(llm_task(SYS_PERFIL,
                                    f'Grupo "{rot}":\n{objs}\n\nDescreva.'))
        if out:
            return out
        if INTERVALO_LLM_SEG:
            time.sleep(INTERVALO_LLM_SEG)
    return {}

perfil_eng = perfil('eng_obra'); time.sleep(INTERVALO_LLM_SEG)
perfil_nao = perfil('nao')
print('ENG_OBRA:', json.dumps(perfil_eng, ensure_ascii=False, indent=2))
print('\nNAO:', json.dumps(perfil_nao, ensure_ascii=False, indent=2))
json.dump({'eng_obra': perfil_eng, 'nao': perfil_nao},
          open(f'{PASTA_RESULTADOS}/05_perfis.json', 'w'), ensure_ascii=False, indent=2)
add_md('5. Perfis (LLM)',
f'''**eng_obra:** {perfil_eng.get("resumo","-")}
- léxico: {", ".join(perfil_eng.get("padroes_lexicais",[])[:5])}

**nao:** {perfil_nao.get("resumo","-")}
- léxico: {", ".join(perfil_nao.get("padroes_lexicais",[])[:5])}''')

## 6. Treina o classificador (LogReg + Random Forest)

**Treino:** positivos = eng+obras + clusters `eng_obra`; negativos = geral
distante + clusters `nao`. (Os `duvidoso` ficam de fora — serão pontuados na
Etapa 7.) Features = embeddings SBERT. Comparamos os dois modelos por F1 e
escolhemos o melhor.

In [ ]:
# 6.1 Treina nos CERTEIROS automáticos (eng_obra vs nao)
mask_tr = df['rotulo_treino'].isin(['eng_obra', 'nao']).values
X_tr_all = X_emb[mask_tr]
y_tr_all = (df.loc[mask_tr, 'rotulo_treino'] == 'eng_obra').astype(int).values
print(f'Treino: {mask_tr.sum():,} | eng_obra={y_tr_all.sum():,} | nao={(y_tr_all==0).sum():,}')
print('⚠ Nota: F1 abaixo é no conjunto fácil (eng vs geral-distante) — '
      'serve para escolher o modelo. O número REAL vem da validação manual (etapa 8).')

Xa, Xb, ya, yb = train_test_split(X_tr_all, y_tr_all, test_size=0.2,
                                   random_state=42, stratify=y_tr_all)

modelos = {
    'logreg': LogisticRegression(max_iter=3000, class_weight='balanced',
                                  C=1.0, n_jobs=-1, random_state=42),
    'rf': RandomForestClassifier(n_estimators=300, max_depth=20,
                                  class_weight='balanced', n_jobs=-1, random_state=42),
    'extratrees': ExtraTreesClassifier(n_estimators=300, max_depth=20,
                                  class_weight='balanced', n_jobs=-1, random_state=42),
    'gradboost': GradientBoostingClassifier(random_state=42),
    'svm_rbf': SVC(kernel='rbf', class_weight='balanced', probability=True,
                    random_state=42),
    'knn': KNeighborsClassifier(n_neighbors=15, weights='distance',
                                 metric='cosine', n_jobs=-1),
    'mlp': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400,
                          early_stopping=True, random_state=42),
    'naive_bayes': GaussianNB(),
}
res = {}
for nome, m in modelos.items():
    try:
        m.fit(Xa, ya)
        pred = m.predict(Xb)
        res[nome] = {'modelo': m, 'pred': pred,
                     'f1': f1_score(yb, pred, average='macro'),
                     'f1_eng': f1_score(yb, pred, pos_label=1, zero_division=0)}
        print(f'  {nome:12s}: F1 macro = {res[nome]["f1"]:.3f} | '
              f'F1 eng_obra = {res[nome]["f1_eng"]:.3f}')
    except Exception as e:
        print(f'  {nome:12s}: falhou ({str(e)[:60]})')

melhor_nome = max(res, key=lambda n: res[n]['f1'])
melhor = res[melhor_nome]['modelo']
print(f'\\n🏆 Melhor: {melhor_nome} (F1={res[melhor_nome]["f1"]:.3f})')

# Calibração de probabilidade: SVM/MLP costumam ser SUPER-CONFIANTES (cospem
# prob ~1.0 em massa), tornando o limiar inútil. Recalibra no holdout (Xb,yb)
# para que prob_eng_obra reflita probabilidade real e o limiar passe a separar.
from sklearn.calibration import CalibratedClassifierCV
try:
    melhor = CalibratedClassifierCV(melhor, method='isotonic', cv='prefit').fit(Xb, yb)
    print('Probabilidades calibradas (isotonic, holdout).')
except Exception as e:
    print('Calibração falhou, mantendo modelo original:', str(e)[:80])

In [ ]:
# 6.2 Matriz de confusão (grade adaptável ao nº de modelos)
n = len(res)
ncol = 3
nrow = (n + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax, (nome, r) in zip(axes, res.items()):
    cm = confusion_matrix(yb, r['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['nao','eng_obra'], yticklabels=['nao','eng_obra'])
    marca = ' 🏆' if nome == melhor_nome else ''
    ax.set_title(f'{nome} (F1={r["f1"]:.3f}){marca}')
    ax.set_xlabel('predito'); ax.set_ylabel('real')
for ax in axes[n:]:
    ax.set_visible(False)
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/06_confusao.png', dpi=120, bbox_inches='tight')
plt.show()

add_md('6. Classificador',
'| modelo | F1 macro | F1 eng_obra |\n|---|---|---|\n' +
'\n'.join(f'| {n} | {r["f1"]:.3f} | {r["f1_eng"]:.3f} |'
          for n, r in sorted(res.items(), key=lambda x: -x[1]["f1"])) +
f'\n\n🏆 Melhor: **{melhor_nome}** (F1={res[melhor_nome]["f1"]:.3f})')

## 7. Pontua TODOS os `geral` → ranking de suspeitos (produto final)

Aplica o melhor modelo a **todos** os contratos `geral` (inclusive os que o
filtro PU descartou) e gera a probabilidade de ser engenharia. Esse ranking é
o produto central da pesquisa: a lista priorizada de prováveis
subenquadramentos.

In [ ]:
# 7. Pontua TODOS os "geral" → ranking de suspeitos
LIMIAR = 0.5     # corte inicial; a validação (etapa 8) ajusta para o ideal

mask_g = (df['rotulo'] == 'geral').values
prob_g = melhor.predict_proba(X_emb[mask_g])[:, 1]
df['prob_eng_obra'] = np.nan
df.loc[mask_g, 'prob_eng_obra'] = prob_g

# Prior de domínio: a PUREZA do cluster (proporção de eng+obras confirmados nele)
# mede quão 'engenheira' é a vizinhança semântica. Combinamos a probabilidade do
# modelo com essa pureza para derrubar falsos positivos de clusters quase sem
# engenharia (ex.: manutenção de veículos) e realçar suspeitos em clusters de
# alta pureza (ex.: obras/reformas). score = prob * (0.4 + 0.6*pureza).
try:
    pur = df_desc.set_index('cluster')['pct_certeiros'].to_dict()
except Exception:
    pur = {}
df['cluster_purity'] = df['cluster'].map(pur).fillna(0.0)
df['score_suspeita'] = df['prob_eng_obra'] * (0.4 + 0.6 * df['cluster_purity'])

df['classe_final'] = df['rotulo_treino']
df.loc[mask_g, 'classe_final'] = np.where(prob_g >= LIMIAR, 'eng_obra_pred', 'nao_pred')
df.loc[df['rotulo'].isin(['engenharia','obras']), 'classe_final'] = 'eng_obra'

# Ranking pelo score combinado (prob x pureza) — mais preciso que prob isolada.
ranking = (df[mask_g].sort_values('score_suspeita', ascending=False)
           [['numeroControlePNCP','text','cluster','prob_eng_obra',
             'cluster_purity','score_suspeita','valor']]).copy()
ranking['cluster_purity'] = ranking['cluster_purity'].round(3)
ranking['score_suspeita'] = ranking['score_suspeita'].round(4)
# Probabilidade legível + salva em formato pt-BR (Excel não estraga os decimais)
ranking['prob_eng_obra'] = ranking['prob_eng_obra'].round(4)
ranking.to_csv(f'{PASTA_RESULTADOS}/07_ranking_suspeitos.csv',
               index=False, encoding='utf-8-sig', sep=';', decimal=',')

n05 = int((prob_g >= 0.5).sum()); n08 = int((prob_g >= 0.8).sum()); n09 = int((prob_g >= 0.9).sum())
print(f'Geral pontuados: {mask_g.sum():,}')
print(f'  prob >= 0.5: {n05:,}   prob >= 0.8: {n08:,}   prob >= 0.9: {n09:,}')
print('\\nTop 15 suspeitos:')
print(ranking.head(15)[['numeroControlePNCP','text','prob_eng_obra']].to_string(index=False))

add_md('7. Ranking de suspeitos',
f'''- Geral pontuados: **{int(mask_g.sum()):,}**
- prob ≥ 0.5: {n05:,} | prob ≥ 0.8: {n08:,} | prob ≥ 0.9: {n09:,}
- CSV (pt-BR, `;` e vírgula decimal): `07_ranking_suspeitos.csv`''')

In [ ]:
# 7.2 Visualizações do ranking: distribuição da probabilidade + valor por órgão
from IPython.display import display

g_df = df[mask_g].copy()   # só os "geral" pontuados

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# (A) Histograma da probabilidade de ser engenharia.
# Boa prática: marcar o limiar de decisão e anotar quantos ficam acima.
ax1.hist(g_df['prob_eng_obra'], bins=50, color='#4c72b0', edgecolor='white')
ax1.axvline(LIMIAR, color='#d62728', ls='--', lw=1.5,
            label=f'limiar = {LIMIAR}')
n_acima = int((g_df['prob_eng_obra'] >= LIMIAR).sum())
ax1.set_title(f'Probabilidade de ser engenharia (n={len(g_df):,} "geral")\\n'
              f'{n_acima:,} acima do limiar')
ax1.set_xlabel('prob_eng_obra'); ax1.set_ylabel('nº de contratos'); ax1.legend()

# (B) Top órgãos por valor agregado dos suspeitos (impacto financeiro).
sus = g_df[g_df['prob_eng_obra'] >= LIMIAR].copy()
if 'valor' in sus.columns and 'razaoSocialOrgao' in sus.columns and len(sus):
    sus['valor'] = pd.to_numeric(sus['valor'], errors='coerce')
    topv = (sus.groupby('razaoSocialOrgao')['valor'].sum()
            .sort_values(ascending=False).head(10))
    ax2.barh([o[:38] for o in topv.index[::-1]], topv.values[::-1] / 1e6,
             color='#dd8452')
    ax2.set_xlabel('valor agregado (R$ milhões)')
    ax2.set_title('Top 10 órgãos por valor de suspeitos')
else:
    ax2.set_visible(False)
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/07_ranking_viz.png', dpi=120, bbox_inches='tight')
plt.show()

# head do ranking bem visível
print('Top 10 do ranking de suspeitos:')
display(ranking.head(10))

## 8. Validação manual (rigor acadêmico)

Sorteamos ~80 `geral` estratificados por faixa de probabilidade para você
rotular à mão. Comparando com o modelo, obtemos **precisão/recall/F1 honestos**
— sem isso, não há número confiável de desempenho para o TCC.

In [ ]:
# 8.1 Amostra de validação estratificada por faixa de prob
# IMPORTANTE: você só precisa preencher a coluna rotulo_verdade (eng_obra|nao).
# NÃO mexa na coluna prob — ela é re-lida da memória, então mesmo que o Excel
# estrague o número, a validação continua correta.
N_VALID = 100
g = df[mask_g].copy()
g['faixa'] = pd.cut(g['prob_eng_obra'], bins=[0,.2,.4,.6,.8,1.0],
                    labels=['0-20','20-40','40-60','60-80','80-100'])
val = (g.groupby('faixa', group_keys=False, observed=True)
       .apply(lambda x: x.sample(min(len(x), N_VALID//5), random_state=42)))
val = val[['numeroControlePNCP','text','prob_eng_obra','faixa']].copy()
val['prob_eng_obra'] = val['prob_eng_obra'].round(4)
val['rotulo_verdade'] = ''     # <-- VOCÊ PREENCHE: eng_obra | nao
cam_val = f'{PASTA_RESULTADOS}/08_validacao.csv'
# Formato pt-BR para o Excel não corromper os números
val.to_csv(cam_val, index=False, encoding='utf-8-sig', sep=';', decimal=',')
print(f'Amostra de validação: {len(val)} contratos → {cam_val}')
print('→ Abra no Excel (separador ; já é o padrão pt-BR), preencha SÓ a coluna')
print('  rotulo_verdade com eng_obra ou nao, salve como CSV, e rode a próxima célula.')

### ⏸ PAUSE — rotule `08_validacao.csv` (coluna `rotulo_verdade`), salve, e rode abaixo

In [ ]:
# 8.2 Mede desempenho REAL e escolhe o melhor LIMIAR de probabilidade
# Robusto a Excel: re-lê a prob LIMPA da memória (df) pelo numeroControlePNCP;
# do arquivo do usuário usa SÓ a coluna rotulo_verdade. Lê tanto ';'/vírgula
# quanto ','/ponto.
import io
_raw = open(cam_val, encoding='utf-8-sig').read()
_sep = ';' if _raw.split('\\n', 1)[0].count(';') >= _raw.split('\\n', 1)[0].count(',') else ','
vdf = pd.read_csv(io.StringIO(_raw), sep=_sep, dtype=str)
vdf.columns = [c.strip() for c in vdf.columns]
vdf['rotulo_verdade'] = vdf['rotulo_verdade'].fillna('').str.strip().str.lower()
vdf = vdf[vdf['rotulo_verdade'].isin(['eng_obra', 'nao'])]

# Re-junta a probabilidade LIMPA da memória (ignora a prob do arquivo)
prob_map = dict(zip(df['numeroControlePNCP'].astype(str), df['prob_eng_obra']))
vdf['prob'] = vdf['numeroControlePNCP'].astype(str).map(prob_map)
vdf = vdf.dropna(subset=['prob'])

if len(vdf) < 10:
    print(f'⚠ Só {len(vdf)} linhas válidas. Preencha rotulo_verdade '
          f'(eng_obra|nao) em {cam_val} e rode de novo.')
else:
    y_true = (vdf['rotulo_verdade'] == 'eng_obra').astype(int).values
    probs = vdf['prob'].values
    print(f'Validação manual: {len(vdf)} contratos '
          f'({y_true.sum()} eng_obra, {(y_true==0).sum()} nao)\\n')
    # Varre limiares e mostra precisão/recall/F1
    print(f'{"limiar":>6} {"prec":>6} {"recall":>7} {"F1":>6}')
    linhas_tab, melhor_lim, melhor_f1 = [], 0.5, -1
    for t in np.arange(0.30, 0.96, 0.05):
        yp = (probs >= t).astype(int)
        P = precision_score(y_true, yp, zero_division=0)
        R = recall_score(y_true, yp, zero_division=0)
        F = f1_score(y_true, yp, zero_division=0)
        print(f'{t:6.2f} {P:6.1%} {R:7.1%} {F:6.3f}')
        linhas_tab.append((t, P, R, F))
        if F > melhor_f1:
            melhor_f1, melhor_lim = F, float(t)

    # Gráfico: precisão, recall e F1 em função do limiar (valida a decisão)
    _t = [x[0] for x in linhas_tab]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(_t, [x[1] for x in linhas_tab], 'o-', label='precisão', color='#2ca02c')
    ax.plot(_t, [x[2] for x in linhas_tab], 's-', label='recall', color='#1f77b4')
    ax.plot(_t, [x[3] for x in linhas_tab], '^-', label='F1', color='#d62728')
    ax.axvline(melhor_lim, color='gray', ls='--', label=f'melhor = {melhor_lim:.2f}')
    ax.set_xlabel('limiar de probabilidade'); ax.set_ylabel('métrica')
    ax.set_title('Validação: precisão × recall × F1 por limiar'); ax.legend()
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/08_curva_limiar.png', dpi=120, bbox_inches='tight')
    plt.show()
    LIMIAR = round(melhor_lim, 2)   # ATUALIZA o limiar global p/ etapas 10 e 11
    Pf = precision_score(y_true, (probs >= LIMIAR).astype(int), zero_division=0)
    Rf = recall_score(y_true, (probs >= LIMIAR).astype(int), zero_division=0)
    print(f'\\n🎯 Melhor limiar = {LIMIAR} (F1={melhor_f1:.3f}, '
          f'precisão={Pf:.1%}, recall={Rf:.1%})')
    n_susp = int((df.loc[mask_g, 'prob_eng_obra'] >= LIMIAR).sum())
    print(f'   Com esse limiar, {n_susp:,} suspeitos no ranking.')
    add_md('8. Validação manual',
f'''- Rotulados à mão: {len(vdf)}
- **Melhor limiar = {LIMIAR}** → precisão **{Pf:.1%}**, recall **{Rf:.1%}**, F1 **{melhor_f1:.3f}**
- Suspeitos com prob ≥ {LIMIAR}: **{n_susp:,}**

| limiar | precisão | recall | F1 |
|---|---|---|---|
''' + '\\n'.join(f'| {t:.2f} | {P:.1%} | {R:.1%} | {F:.3f} |'
                 for t, P, R, F in linhas_tab))

## 9. Visualização UMAP 2D

Projeta os embeddings em 2D. Cores por `classe_final`. Se os `geral` preditos
como engenharia (verde claro) caem junto dos certeiros eng (verde escuro), o
modelo concorda geometricamente.

In [ ]:
import umap
MAX_VIZ = 8000
idx_v = (np.random.RandomState(42).choice(len(df), MAX_VIZ, replace=False)
         if len(df) > MAX_VIZ else np.arange(len(df)))
print('Projetando UMAP...')
emb2d = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                  random_state=42).fit_transform(X_emb[idx_v])

dv = df.iloc[idx_v].copy()
dv['x'], dv['y'] = emb2d[:,0], emb2d[:,1]
paleta = {'eng_obra':'#1b7837','nao':'#b2182b',
          'eng_obra_pred':'#7fbc41','nao_pred':'#ef8a62',
          'duvidoso':'#cccccc'}
fig, ax = plt.subplots(figsize=(11, 8))
for cl, cor in paleta.items():
    s = dv[dv['classe_final'] == cl]
    if len(s):
        ax.scatter(s['x'], s['y'], c=cor, s=8, alpha=.55,
                   label=f'{cl} ({len(s)})', edgecolors='none')
ax.legend(); ax.set_title(f'UMAP 2D — classe final (modelo: {melhor_nome})')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/09_umap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 9.1 UMAP interativo (Plotly) — passe o mouse para inspecionar cada contrato.
# Use para CAÇAR suspeitos: hover mostra objeto, órgão, prob e nº de controle.
# O PNG estático (célula anterior) fica para o relatório; este é exploratório.
import plotly.express as px

dvp = dv.copy()
dvp['objeto_curto'] = dvp['objeto'].astype(str).str.slice(0, 90)
hover_cols = {c: True for c in ['razaoSocialOrgao', 'prob_eng_obra',
                                'numeroControlePNCP'] if c in dvp.columns}
hover_cols['objeto_curto'] = True
hover_cols['x'] = False; hover_cols['y'] = False

figp = px.scatter(
    dvp, x='x', y='y', color='classe_final',
    color_discrete_map={'eng_obra':'#1b7837', 'nao':'#b2182b',
                        'eng_obra_pred':'#7fbc41', 'nao_pred':'#ef8a62',
                        'duvidoso':'#cccccc'},
    hover_data=hover_cols, opacity=0.6,
    title=f'UMAP interativo — classe final (modelo: {melhor_nome})')
figp.update_traces(marker=dict(size=5))
figp.update_layout(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   legend_title_text='classe', height=650)
# Salva como HTML interativo (abre no navegador, ótimo para a banca explorar).
figp.write_html(f'{PASTA_RESULTADOS}/09_umap_interativo.html')
figp.show()


In [ ]:
# 9.2 Grafo de similaridade k-NN (visualização de dados complexos como rede)
# Cada nó é um contrato; arestas ligam cada contrato aos k vizinhos mais
# parecidos (k-NN no espaço de embeddings). Posição = projeção UMAP já
# calculada. Cor = classe final. Comunidades densas de "geral predito eng"
# coladas aos "engenharia confirmados" reforçam o subenquadramento.
import networkx as nx
from sklearn.neighbors import kneighbors_graph

# Usa a MESMA amostra/posições da UMAP (idx_v, emb2d, df_viz) para coerência.
Xv = X_emb[idx_v]
K_GRAFO = 4
A = kneighbors_graph(Xv, n_neighbors=K_GRAFO, metric='cosine', mode='connectivity')
G = nx.from_scipy_sparse_array(A)
pos = {i: (emb2d[i, 0], emb2d[i, 1]) for i in range(len(idx_v))}

paleta = {'eng_obra':'#1b7837','nao':'#b2182b',
          'eng_obra_pred':'#7fbc41','nao_pred':'#ef8a62'}
cores_no = [paleta.get(cf, '#cccccc') for cf in df_viz['classe_final'].values]

fig, ax = plt.subplots(figsize=(11, 9))
# arestas bem leves para não poluir (boa prática: reduzir "tinta" supérflua)
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.08, width=0.4, edge_color='#888')
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=14, node_color=cores_no,
                       linewidths=0)
# legenda manual
from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, label=k, markersize=8)
       for k, c in paleta.items()]
ax.legend(handles=leg, loc='best', frameon=True)
ax.set_title(f'Rede de similaridade k-NN (k={K_GRAFO}) sobre {len(idx_v):,} contratos\\n'
             'nós colados = objetos semanticamente parecidos')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/09_grafo_knn.png', dpi=130, bbox_inches='tight')
plt.show()

# Detecção de comunidades (informativo): tamanho das maiores comunidades
try:
    comms = nx.community.greedy_modularity_communities(G)
    print(f'{len(comms)} comunidades; 5 maiores têm '
          f'{[len(c) for c in comms[:5]]} nós')
except Exception as e:
    print('comunidades:', str(e)[:80])

## 10. Veredito LLM nos suspeitos

A LLM revisa os top suspeitos (regra de ouro: locação/aquisição/evento = não)
como dupla checagem antes da análise de rito.

In [ ]:
# 10. Veredito LLM nos suspeitos (checagem cruzada, regra de ouro)
SYS_VER = '''Você é engenheiro analista (Lei 14.133/2021). Pelo OBJETO, diga se o
contrato envolve ENGENHARIA/OBRAS (atividade que exige ART/CREA, projeto e
responsável técnico).

CONTA como eng_obra (intervenção física / serviço técnico de engenharia):
construção, reforma, ampliação, demolição, pavimentação, obra; manutenção
PREDIAL/civil; instalação ou manutenção elétrica, hidráulica, estrutural ou de
climatização; impermeabilização, pintura predial, troca de piso/revestimento/
telhado/esquadrias; drenagem, terraplanagem, urbanização.

NAO conta (é "nao"): aquisição/fornecimento de bens, locação de equipamento,
evento/show/curso, serviços administrativos ou de mão de obra, alimentação,
transporte/vale, energia/água, seguros, limpeza/jardinagem comum, manutenção de
veículos.

Na dúvida entre serviço comum e intervenção física construtiva, marque eng_obra
com confiança baixa (não descarte reformas/manutenção predial).
Responda JSON: {"classe":"eng_obra"|"nao","confianca":0.0-1.0,"motivo":"até 30 palavras"}'''

# Usa o LIMIAR ajustado pela validação; revisa até N_VEREDITO suspeitos
N_VEREDITO = 60
top_susp = (df[mask_g & (df['prob_eng_obra'] >= LIMIAR)]
            .sort_values('score_suspeita', ascending=False).head(N_VEREDITO))
print(f'LLM revisando {len(top_susp)} suspeitos (prob ≥ {LIMIAR})...')
ver = []
for i, (_, row) in enumerate(top_susp.iterrows()):
    if i and INTERVALO_LLM_SEG: time.sleep(INTERVALO_LLM_SEG)
    p = extrair_json(llm_task(SYS_VER, f"Objeto: {str(row['text'])[:500]}")) or {}
    ver.append({'numeroControlePNCP': row['numeroControlePNCP'],
                'prob_modelo': round(float(row['prob_eng_obra']), 3),
                'llm_classe': p.get('classe','?'),
                'llm_conf': float(p.get('confianca',0) or 0),
                'llm_motivo': str(p.get('motivo',''))[:200],
                'objeto': str(row['text'])[:160]})
    if (i+1) % 15 == 0: print(f'  {i+1}/{len(top_susp)}')

df_ver = pd.DataFrame(ver)
df_ver['confirmado'] = (df_ver['llm_classe']=='eng_obra') & (df_ver['llm_conf']>=0.6)
n_conf = int(df_ver['confirmado'].sum())
df_ver.to_csv(f'{PASTA_RESULTADOS}/10_veredito_llm.csv',
              index=False, encoding='utf-8-sig', sep=';', decimal=',')
print(f'\\nLLM confirmou {n_conf}/{len(df_ver)} como engenharia.')
add_md('10. Veredito LLM',
       f'- {len(df_ver)} suspeitos revisados → **{n_conf}** confirmados como engenharia')
df_ver.head(20)

## 11. Análise de RITO (evidência definitiva)

Para os suspeitos **confirmados pelo LLM**, baixamos os documentos da
**licitação (compra)** — é onde ficam o TR, Projeto Básico e Edital (o contrato
em si não os expõe). Detectamos marcadores legais (ART/CREA, projeto básico,
ABNT, etc.) e pedimos um veredito à LLM sobre o trecho do TR.

**Descoberta-chave (Manual API PNCP §4.1):** o `numeroControlePNCP` tem dígito
1 = contratação (compra) e 2 = contrato. O TR/PB pertencem à compra,
referenciada pelo campo `numeroControlePNCPCompra`.

**Veredito final:**
- `subenquadramento_real` — PDF legível SEM rito → provável violação da Lei 14.133
- `rotulacao_incorreta_processo_ok` — rito presente → erro de cadastro
- `indeterminado_*` — compra não resolvida / sem documento / PDF ilegível

Tudo inline (sem dependências externas além de `requests` + PyMuPDF).

In [ ]:
# 11.1 Funções inline de download + marcadores (copiadas do pipeline validado)
import requests

MAX_PDFS_BAIXAR = 200         # limite de contratos (cada um leva ~5-20s)
MAX_DOCS_POR_CONTRATO = 3
LIMIAR_RITO = LIMIAR    # usa o limiar ajustado pela validação (etapa 8)
CACHE_PDFS = Path(PASTA_RESULTADOS) / 'cache_pdfs'
CACHE_PDFS.mkdir(parents=True, exist_ok=True)
_API = 'https://pncp.gov.br/api/pncp/v1'
_HEADERS = {'User-Agent': 'Mozilla/5.0 (pesquisa academica PNCP)'}
_RX_NCP = re.compile(r'^(?P<cnpj>\d{14})-(?P<tipo>\d+)-(?P<seq>\d+)/(?P<ano>\d{4})$')

def _decompor_ncp(num):
    m = _RX_NCP.match(str(num).strip()) if num else None
    return ({'cnpj': m['cnpj'], 'tipo': int(m['tipo']),
             'ano': int(m['ano']), 'sequencial': int(m['seq'])} if m else None)

# Marcadores de RITO DE ENGENHARIA — apenas CREA/ART (sem CAU/RRT)
MARCADORES = {
    'ART': [r'\banota[çc][ãa]o\s+de\s+responsabilidade\s+t[ée]cnica\b',
            r'\bART\b(?:\s+do\s+CREA)?'],
    'CREA': [r'\bCREA[/\s\-]?\w{0,2}\b',
             r'\bConselho\s+Regional\s+de\s+Engenharia\b'],
    'ENGENHEIRO_RESP': [r'\bengenheir[oa]?\s+respons[áa]vel\b',
                         r'\brespons[áa]vel\s+t[ée]cnico\b'],
    'ATESTADO_CAP_TEC': [r'\batestado\s+de\s+capacidade\s+t[ée]cnica\b'],
    'PROJETO_BASICO': [r'\bprojeto\s+b[áa]sico\b', r'\bprojeto\s+executivo\b',
                        r'\bmemorial\s+descritivo\b'],
    'OBRA_SERV_ENG': [r'\bobra\s+de\s+engenharia\b',
                       r'\bservi[çc]os?\s+de\s+engenharia\b'],
    'ABNT_NORMA': [r'\bABNT\s+NBR\s*\d+'],
    'LEI_14133_ENG': [r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XII\b',
                       r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XX(?:I+)?\b'],
}
_RX_MARC = {n: [re.compile(p, re.IGNORECASE) for p in ps] for n, ps in MARCADORES.items()}

def detectar_marcadores(texto):
    if not texto:
        return {n: False for n in MARCADORES}
    return {n: any(rx.search(texto) for rx in lst) for n, lst in _RX_MARC.items()}

def normalizar_pdf_text(t):
    if not t:
        return ''
    t = re.sub(r'-\s*\n\s*', '', t)
    t = re.sub(r'(?<!\n)\n(?!\n)', ' ', t)
    return re.sub(r'[ \t]+', ' ', t)

def extrair_texto_pdf(caminho, max_pag=30):
    try:
        import fitz
        doc = fitz.open(caminho)
        txt = []
        for i, p in enumerate(doc):
            if i >= max_pag:
                break
            txt.append(p.get_text())
        doc.close()
        return normalizar_pdf_text('\n'.join(txt))
    except Exception:
        return ''

_RESOLVE_DBG = []   # guarda as 1as respostas cruas da API p/ depurar o vinculo

def _achar_compra_em(d):
    '''Procura, tolerante a maiusc/minusc, o vinculo da compra dentro do JSON de
    detalhe do contrato - o nome do campo varia entre endpoints do PNCP.'''
    if not isinstance(d, dict):
        return None
    # (1) qualquer chave com 'compra' cujo valor seja um numeroControlePNCP
    for k, v in d.items():
        if 'compra' in k.lower() and isinstance(v, str) and _RX_NCP.match(v.strip()):
            info = _decompor_ncp(v)
            if info:
                return info
    # (2) sequencialCompra + anoCompra (+ cnpj do orgao)
    low = {k.lower(): v for k, v in d.items()}
    sq, an = low.get('sequencialcompra'), low.get('anocompra')
    if sq and an:
        cj = ((d.get('orgaoEntidade') or {}).get('cnpj')
              or (d.get('orgaoSubRogado') or {}).get('cnpj'))
        if cj:
            return {'cnpj': re.sub(r'\D', '', str(cj)), 'tipo': 1,
                    'ano': int(an), 'sequencial': int(sq)}
    # (3) sub-objeto aninhado
    for chave in ('compra', 'compraOrigem', 'contratacao'):
        if isinstance(d.get(chave), dict):
            r = _achar_compra_em(d[chave])
            if r:
                return r
    return None

def _resolver_compra(ncp_contrato, row):
    # (a) coluna do parquet - caminho rapido, sem API
    nc = row.get('numeroControlePNCPCompra')
    if isinstance(nc, str) and nc.strip():
        info = _decompor_ncp(nc)
        if info:
            return info
    info_c = _decompor_ncp(ncp_contrato)
    if not info_c:
        return None
    # (b) se o proprio NCP ja e contratacao (tipo 1), usa direto
    if info_c['tipo'] == 1:
        return info_c
    # (c) consulta o detalhe do contrato e procura o vinculo da compra
    cnpj, ano, seq = info_c['cnpj'], info_c['ano'], info_c['sequencial']
    url = f'{_API}/orgaos/{cnpj}/contratos/{ano}/{seq}'
    r = d = None
    try:
        r = requests.get(url, timeout=25, headers=_HEADERS)
        d = r.json() if r.status_code == 200 else None
    except Exception:
        pass
    if len(_RESOLVE_DBG) < 8:
        _RESOLVE_DBG.append({'ncp': ncp_contrato, 'url': url,
            'status': (r.status_code if r is not None else 'ERR'),
            'keys': (list(d.keys())[:18] if isinstance(d, dict) else None)})
    return _achar_compra_em(d) if d else None

def _listar_docs(info):
    try:
        r = requests.get(f'{_API}/orgaos/{info["cnpj"]}/compras/'
                         f'{info["ano"]}/{info["sequencial"]}/arquivos',
                         timeout=20, headers=_HEADERS)
        if r.status_code != 200:
            return []
        d = r.json()
        return d if isinstance(d, list) else d.get('data', [])
    except Exception:
        return []

def _baixar(d, info, destino):
    urls = [d[c] for c in ('url','uri','urlArquivo','link') if d.get(c)]
    sq = (d.get('sequencialDocumento') or d.get('sequencial') or d.get('id'))
    if sq:
        urls.append(f'{_API}/orgaos/{info["cnpj"]}/compras/'
                    f'{info["ano"]}/{info["sequencial"]}/arquivos/{sq}')
    for u in urls:
        try:
            r = requests.get(u, timeout=60, stream=True, headers=_HEADERS)
            if r.status_code == 200:
                with open(destino, 'wb') as f:
                    for ch in r.iter_content(8192):
                        f.write(ch)
                if destino.stat().st_size > 0:
                    return True
        except Exception:
            continue
    return False

print('funções de rito definidas')

In [ ]:
# 11.1b ENRIQUECIMENTO EM LOTE: preenche numeroControlePNCPCompra só dos suspeitos
# Resolve o gargalo do parquet antigo (sem a coluna da compra) SEM re-coletar os
# 30 mil. Para os suspeitos do ranking, consulta o detalhe do contrato uma vez,
# extrai o vínculo da compra e GRAVA EM CACHE (enriquecimento_compra.csv) — assim
# re-execuções são instantâneas e o loop de rito (11.2) usa o caminho rápido (a).
CACHE_COMPRA = Path(PASTA_RESULTADOS) / 'enriquecimento_compra.csv'

# alvo = mesmos suspeitos que entram no rito (geral, prob>=limiar, por score)
_alvo_enr = (df[mask_g & (df['prob_eng_obra'] >= LIMIAR_RITO)]
             .sort_values('score_suspeita', ascending=False).head(MAX_PDFS_BAIXAR))
_ncps = _alvo_enr['numeroControlePNCP'].astype(str).tolist()
print(f'Enriquecendo {len(_ncps)} suspeitos com o vínculo da compra...')

# garante a coluna no df + carrega cache existente
if 'numeroControlePNCPCompra' not in df.columns:
    df['numeroControlePNCPCompra'] = ''
df['numeroControlePNCPCompra'] = df['numeroControlePNCPCompra'].fillna('').astype(str)
mapa = {}
if CACHE_COMPRA.exists():
    _cc = pd.read_csv(CACHE_COMPRA, sep=';', dtype=str).fillna('')
    mapa = dict(zip(_cc['numeroControlePNCP'], _cc['numeroControlePNCPCompra']))
    print(f'  cache: {len(mapa)} já resolvidos anteriormente')

_ok = _fail = _api = 0
for ncp in _ncps:
    if mapa.get(ncp):                      # já no cache (não vazio)
        _ok += 1; continue
    # tenta resolver via API de detalhe (mesma lógica robusta da 11.1)
    info = _decompor_ncp(ncp)
    achou = ''
    if info and info['tipo'] == 1:
        achou = f'{info["cnpj"]}-1-{info["sequencial"]:06d}/{info["ano"]}'
    elif info:
        _api += 1
        url = f'{_API}/orgaos/{info["cnpj"]}/contratos/{info["ano"]}/{info["sequencial"]}'
        d = None; r = None
        try:
            r = requests.get(url, timeout=25, headers=_HEADERS)
            d = r.json() if r.status_code == 200 else None
        except Exception:
            d = None
        if len(_RESOLVE_DBG) < 8:
            _RESOLVE_DBG.append({'ncp': ncp, 'url': url,
                'status': (r.status_code if r is not None else 'ERR'),
                'keys': (list(d.keys())[:18] if isinstance(d, dict) else None)})
        ic = _achar_compra_em(d) if d else None
        if ic:
            achou = f'{ic["cnpj"]}-1-{ic["sequencial"]:06d}/{ic["ano"]}'
        time.sleep(0.15)
    mapa[ncp] = achou
    if achou: _ok += 1
    else:     _fail += 1

# grava cache e injeta no df (caminho rápido do _resolver_compra)
pd.DataFrame([{'numeroControlePNCP': k, 'numeroControlePNCPCompra': v}
              for k, v in mapa.items()]).to_csv(CACHE_COMPRA, index=False, sep=';')
df['numeroControlePNCPCompra'] = (df['numeroControlePNCP'].astype(str).map(mapa)
                                  .fillna(df['numeroControlePNCPCompra']))
print(f'  resolvidos: {_ok} | sem vínculo: {_fail} | chamadas API: {_api}')
if _fail and _RESOLVE_DBG:
    print('  amostra de respostas cruas (diagnóstico):')
    for dbg in _RESOLVE_DBG[:5]:
        print('   ', dbg)
if _fail > 0.5 * max(1, len(_ncps)):
    print('  ⚠ Maioria sem vínculo via API → caminho garantido é re-coletar o '
          'parquet com a versão nova de pncp/coleta.py (salva esse campo direto).')


In [ ]:
# 11.2 Baixa e analisa o rito — entrada vinda do RANKING (não só do veredito)
SYS_RITO = '''Você analisa um TRECHO de Termo de Referência / Projeto Básico /
Edital de licitação pública. Diga se evidencia RITO DE ENGENHARIA: exigência de
ART do CREA, projeto básico/executivo, engenheiro responsável, atestado de
capacidade técnica, memorial descritivo, planilha orçamentária, normas ABNT, ou
artigos 6º XII/XX/XXI da Lei 14.133/2021.
Responda JSON: {"rito":"sim"|"nao"|"parcial","evidencias":["..."],"confianca":0.0-1.0}'''

# Seleção: TODOS os suspeitos do ranking com prob >= LIMIAR_RITO, até o teto.
# (Antes dependia dos confirmados do veredito = no máx. ~40; por isso o rito
#  trazia quase nada. Agora usa o ranking inteiro até MAX_PDFS_BAIXAR.)
alvo = (df[mask_g & (df['prob_eng_obra'] >= LIMIAR_RITO)]
        .sort_values('prob_eng_obra', ascending=False).head(MAX_PDFS_BAIXAR))
ncp_confirmados_llm = set(df_ver[df_ver['confirmado']]['numeroControlePNCP'].astype(str)) \
    if 'df_ver' in dir() else set()
print(f'Analisando rito de {len(alvo)} suspeitos (prob ≥ {LIMIAR_RITO}, '
      f'teto {MAX_PDFS_BAIXAR})...')

tem_col_compra = 'numeroControlePNCPCompra' in df.columns
if not tem_col_compra:
    print('⚠ parquet SEM coluna numeroControlePNCPCompra (coleta antiga) → '
          'vou resolver a compra via API contrato-a-contrato (mais lento).')

rito = []
diag = Counter()    # contadores de diagnóstico
n_baix = n_diag = 0
for i, (_, row) in enumerate(alvo.iterrows()):
    ncp = str(row['numeroControlePNCP'])
    reg = {'numeroControlePNCP': ncp, 'objeto': str(row['text'])[:200],
           'prob_eng_obra': round(float(row.get('prob_eng_obra') or 0), 3),
           'llm_confirmou': ncp in ncp_confirmados_llm,
           'ncp_compra': '', 'n_pdfs': 0, 'chars': 0, 'mk_score': 0,
           'llm_rito': '', 'llm_conf': 0.0}
    info_c = _resolver_compra(ncp, row)
    if not info_c:
        diag['sem_compra'] += 1
        reg['classificacao_rito'] = 'indeterminado_sem_compra'
        rito.append(reg); continue
    diag['compra_resolvida'] += 1
    reg['ncp_compra'] = f'{info_c["cnpj"]}-1-{info_c["sequencial"]:06d}/{info_c["ano"]}'
    docs = _listar_docs(info_c)
    if n_diag < 5:
        print(f'  • {ncp} → compra {reg["ncp_compra"]} docs={len(docs)}'
              + (f' campos={list(docs[0].keys())[:6]}' if docs else '')); n_diag += 1
    if not docs:
        diag['sem_documento'] += 1
        reg['classificacao_rito'] = 'indeterminado_sem_documento'
        rito.append(reg); continue
    diag['com_documento'] += 1
    textos = []
    for d in docs[:MAX_DOCS_POR_CONTRATO]:
        sq = d.get('sequencialDocumento') or d.get('sequencial') or d.get('id') or len(textos)
        dest = CACHE_PDFS / f'{reg["ncp_compra"].replace("/","_")}_{sq}.pdf'
        if not (dest.exists() and dest.stat().st_size > 0):
            if not _baixar(d, info_c, dest):
                continue
            n_baix += 1; time.sleep(0.2)
        textos.append(extrair_texto_pdf(dest))
    texto = '\\n'.join(t for t in textos if t)
    reg['n_pdfs'] = len(textos); reg['chars'] = len(texto)
    if len(texto) < 300:
        diag['pdf_ilegivel'] += 1
        reg['classificacao_rito'] = 'indeterminado_pdf_ilegivel'
        rito.append(reg); continue
    diag['lido_ok'] += 1
    marc = detectar_marcadores(texto)
    for c, v in marc.items():
        reg[f'mk_{c}'] = bool(v)
    reg['mk_score'] = sum(1 for v in marc.values() if v)
    if INTERVALO_LLM_SEG: time.sleep(INTERVALO_LLM_SEG)
    pr = extrair_json(llm_task(SYS_RITO, f'Documento:\\n{texto[:5000]}')) or {}
    reg['llm_rito'] = pr.get('rito', ''); reg['llm_conf'] = float(pr.get('confianca',0) or 0)
    if reg['mk_score'] >= 2 or pr.get('rito') == 'sim':
        reg['classificacao_rito'] = 'rotulacao_incorreta_processo_ok'
    else:
        reg['classificacao_rito'] = 'subenquadramento_real'
    rito.append(reg)
    if (i+1) % 25 == 0:
        print(f'  {i+1}/{len(alvo)} | {dict(diag)} baixados={n_baix}')

df_rito = pd.DataFrame(rito)
df_rito.to_csv(f'{PASTA_RESULTADOS}/11_analise_rito.csv',
               index=False, encoding='utf-8-sig', sep=';', decimal=',')
print(f'\\n=== Diagnóstico do rito ({len(df_rito)} contratos) ===')
for k in ('compra_resolvida','sem_compra','com_documento','sem_documento',
          'lido_ok','pdf_ilegivel'):
    print(f'  {k:18s}: {diag.get(k,0)}')
print('\\nVeredito:')
print(df_rito['classificacao_rito'].value_counts().to_string())
if diag.get('sem_compra',0) > 0.3*len(df_rito):
    print('\nMuitos "sem_compra". Respostas CRUAS da API de detalhe do '
          'contrato (revelam o nome real do campo da compra):')
    for dbg in _RESOLVE_DBG:
        print('   ', dbg)
    print('\nLeitura: status != 200 -> endpoint/headers; status 200 mas "keys" '
          'sem nada de compra -> o detalhe nao traz o vinculo nesta API. '
          'Caminho GARANTIDO: re-colete com a versao nova de pncp/coleta.py, que '
          'ja salva numeroControlePNCPCompra (resolucao vira instantanea, sem API).')


In [ ]:
# 11.3 Gráfico do veredito de rito
if len(df_rito):
    fig, ax = plt.subplots(figsize=(8, 4))
    cont = df_rito['classificacao_rito'].value_counts()
    cr = {'subenquadramento_real':'#d62728','rotulacao_incorreta_processo_ok':'#ff7f0e',
          'indeterminado_sem_compra':'#9aa0a6','indeterminado_sem_documento':'#bdbdbd',
          'indeterminado_pdf_ilegivel':'#dddddd'}
    ax.barh(cont.index.astype(str)[::-1], cont.values[::-1],
            color=[cr.get(c,'#1f77b4') for c in cont.index[::-1]])
    for i, v in enumerate(cont.values[::-1]):
        ax.text(v, i, f' {v}', va='center')
    ax.set_title('Veredito de rito'); ax.set_xlabel('nº contratos')
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/11_rito.png', dpi=120, bbox_inches='tight')
    plt.show()
    n_viol = int((df_rito['classificacao_rito']=='subenquadramento_real').sum())
    print(f'⚠ {n_viol} prováveis violações da Lei 14.133/2021')
    add_md('11. Análise de rito',
f'''- Confirmados analisados: {len(df_rito)} | PDFs baixados: {n_baix}
- **Prováveis violações (subenquadramento_real): {n_viol}**
- Distribuição: {", ".join(f"{k}={v}" for k,v in cont.items())}''')

## 12. Exportação

Salva o **modelo pronto** (`modelo_final.joblib`), os CSVs e o relatório final.
A última célula mostra como classificar um contrato novo.

In [ ]:
import joblib
joblib.dump(melhor, f'{PASTA_RESULTADOS}/modelo_final.joblib')
df[['numeroControlePNCP','text','rotulo','cluster','rotulo_treino',
    'classe_final','prob_eng_obra']].to_csv(
    f'{PASTA_RESULTADOS}/12_df_classificado.csv', index=False, encoding='utf-8-sig')
print('Salvos: modelo_final.joblib, 12_df_classificado.csv')
print('Relatório vivo final:', RELATORIO_PATH)
print('\nArtefatos em', PASTA_RESULTADOS, ':')
for f in sorted(os.listdir(PASTA_RESULTADOS)):
    print('  -', f)

In [ ]:
# 12.x Reuso do modelo: busca o ÚLTIMO MÊS direto da API do PNCP e classifica
# Não precisa do parquet — pega contratos publicados recentemente, gera
# embedding e aplica o modelo treinado. Tudo inline.
import requests

def classificar_objeto(texto):
    emb = sbert.encode([limpar_boilerplate(texto)], normalize_embeddings=True)
    prob = float(melhor.predict_proba(emb)[0, 1])
    return ('eng_obra' if prob >= LIMIAR else 'nao'), round(prob, 3)

def buscar_pncp_periodo(data_ini, data_fim, uf=None, paginas=3, tam=50):
    '''Busca contratos publicados no período via API de consulta do PNCP.
    datas no formato AAAAMMDD. categoria 8 = serviços (onde mora o subenq).'''
    base = 'https://pncp.gov.br/api/consulta/v1/contratos'
    out = []
    for pag in range(1, paginas + 1):
        params = {'dataInicial': data_ini, 'dataFinal': data_fim,
                  'pagina': pag, 'tamanhoPagina': tam}
        try:
            r = requests.get(base, params=params, timeout=30,
                             headers={'User-Agent': 'Mozilla/5.0'})
            if r.status_code != 200:
                break
            data = r.json().get('data', [])
            if not data:
                break
            for c in data:
                if uf and (c.get('unidadeOrgao') or {}).get('ufSigla') != uf:
                    continue
                obj = (c.get('objetoContrato') or '').strip()
                if len(obj) < 20:
                    continue
                out.append({'numeroControlePNCP': c.get('numeroControlePNCP'),
                            'objeto': obj,
                            'categoria': (c.get('categoriaProcesso') or {}).get('nome', '')})
        except Exception as e:
            print('[api]', str(e)[:100]); break
    return pd.DataFrame(out)

# Exemplo: últimos 30 dias (ajuste as datas)
import datetime as _dt
_hoje = _dt.date(2026, 6, 29)     # troque por _dt.date.today() no Colab
_ini = (_hoje - _dt.timedelta(days=30)).strftime('%Y%m%d')
_fim = _hoje.strftime('%Y%m%d')
print(f'Buscando contratos {_ini}–{_fim} no PNCP...')
df_novos = buscar_pncp_periodo(_ini, _fim, uf='SP', paginas=3)
print(f'{len(df_novos)} contratos recentes obtidos.')

if len(df_novos):
    embs = sbert.encode(df_novos['objeto'].map(limpar_boilerplate).tolist(),
                        normalize_embeddings=True, batch_size=64)
    df_novos['prob_eng_obra'] = melhor.predict_proba(embs)[:, 1].round(3)
    df_novos['classe'] = np.where(df_novos['prob_eng_obra'] >= LIMIAR,
                                   'eng_obra', 'nao')
    df_novos = df_novos.sort_values('prob_eng_obra', ascending=False)
    df_novos.to_csv(f'{PASTA_RESULTADOS}/13_novos_classificados.csv',
                    index=False, encoding='utf-8-sig', sep=';', decimal=',')
    print('\\nTop 10 novos suspeitos de subenquadramento:')
    print(df_novos.head(10)[['numeroControlePNCP','objeto','prob_eng_obra']]
          .to_string(index=False))

# Classificação avulsa:
for ex in ['CONTRATAÇÃO DE EMPRESA PARA PAVIMENTAÇÃO ASFÁLTICA DA RUA X',
           'AQUISIÇÃO DE MATERIAL DE ESCRITÓRIO',
           'MANUTENÇÃO ELÉTRICA PREDIAL COM SUBSTITUIÇÃO DE QUADROS']:
    print(classificar_objeto(ex), '←', ex[:55])

---
### Resumo do método

1. **SBERT** representa cada objeto semanticamente
2. **Filtro PU**: eng+obras (confirmados) ancoram a busca; só `geral` próximo é investigado
3. **Clusters coesos** (auto-k) + **sua revisão** (assistida por LLM) geram rótulos limpos
4. **Classificador** (LogReg/RF) aprende com os rótulos limpos
5. **Todos os `geral`** recebem probabilidade → ranking de suspeitos
6. **Validação manual** dá precisão/recall honestos
7. **Rito** (TR/Edital da licitação) traz a evidência definitiva de violação

O `modelo_final.joblib` classifica contratos futuros direto do objeto.

## Sobre as visualizações desta pesquisa

A pesquisa aplica vários conceitos da disciplina de **Visualização de Dados**:

- **Parte I (gráficos simples + elementos visuais):** os gráficos usam cor com
  significado (verde = engenharia, vermelho = não), rótulos diretos nas barras,
  ordenação por valor e remoção de excesso de "tinta" (decluttering — Knaflic,
  Cairo). Ex.: distribuição de rótulos, pureza dos clusters, histograma de
  probabilidade.
- **Parte II (dados complexos como grafos):** a **rede de similaridade k-NN**
  (etapa 9) modela os contratos como um grafo onde nós próximos são objetos
  semanticamente parecidos — exatamente a *Aplicação 2* do curso. Comunidades
  densas de "geral predito engenharia" coladas aos confirmados são a evidência
  visual do subenquadramento.
- **Parte III (big data + IA + validação de qualidade):** projeção **UMAP** de
  ~30 mil embeddings (redução de dimensionalidade para visualização de big
  data) e o **silhouette** como medida quantitativa de qualidade do
  agrupamento; a **curva de limiar** (precisão × recall) valida a decisão do
  modelo visualmente.
- **Parte IV (sistemas/ferramentas):** abordagem **híbrida** — matplotlib/seaborn
  para as figuras estáticas do TCC (vetoriais, reprodutíveis) e **Plotly** para a
  exploração interativa (UMAP com *hover* mostrando objeto/órgão/prob, salvo em
  `09_umap_interativo.html`). Os `.png` + o HTML + `relatorio_vivo.md` formam um
  painel reproduzível.

> **Storytelling:** a sequência de figuras conta a história — distribuição →
> filtro PU → clusters → fronteira do modelo → ranking → rede → rito —
> conduzindo da base bruta até a evidência de subenquadramento.